In [38]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_classic.schema import Document

In [39]:

video_id = "Js05B8Z1ivE"

try:
    transcript = YouTubeTranscriptApi().fetch(video_id)

    text = "\n".join(
            f"[{item.start:.2f}] {item.text}"
            for item in transcript
        )
        
    text = (
    text.replace("\u200b", "")
        .replace("\u200c", "")
        .replace("\u200d", "")
        .replace("\ufeff", "")
)

except Exception as e:
    print(type(e).__name__)
    print(e)

In [40]:
text

'[0.00] In this quick Python tutorial for\n[2.00] beginners, you\'re going to learn all of\n[3.76] the essentials. We\'ll install Python,\n[6.36] write your first program, and walk\n[8.00] through variables, lists, loops,\n[10.52] functions, and all of the core building\n[12.32] blocks, and by the end we\'ll even put it\n[14.04] into a small program that actually does\n[16.32] something. And this is all from scratch,\n[18.24] so if you\'ve never written a line of\n[19.32] code before, don\'t worry. Let\'s dive in.\n[21.92] So, the first thing we need to do is\n[23.28] install Python itself. So, open up your\n[25.52] browser, you can go over to python.org,\n[28.52] and then you want to go to the download\n[30.08] section. For downloads, you can just\n[32.00] find the most recent download here, so\n[33.52] Python install manager, or you can\n[35.64] install the standalone version. Now, I\n[37.24] recommend that you go with the\n[38.16] standalone version because it\'s going to\n[39.60] b

In [41]:
docs = Document(
    page_content=text,
    metadata={
        "source": "Youtube",
        "video_id": video_id
    }
)

In [42]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents([docs])

In [43]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_classic.schema import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os


load_dotenv()

True

In [54]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(
        model='gemini-embedding-2-preview',
        output_dimensionality=768
    ),
    persist_directory='Youtube_cap',
    collection_name='Captions'
)

In [59]:
results = vector_store._collection.get(
    where={"video_id": video_id},
    limit=1
)

# If the current video is NOT in the database,
# remove the old documents and add the new ones.
if not results["ids"]:
    vector_store.delete(where={})      # Deletes all documents

In [60]:
results = vector_store._collection.get(
    where={"video_id": video_id},
    limit=1
)

if results["ids"]:
    print("Video already indexed.")
else:
    vector_store.add_documents(chunks)

Video already indexed.


In [61]:
from langchain_classic.retrievers import MultiQueryRetriever

model_query = GoogleGenerativeAI(
    model = 'gemini-3.6-flash'
)

multiquery_retriver = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(
        search_kwargs = {'k': 5}
    ),
    llm = model_query
)

In [62]:
query = 'How to install python'

In [63]:
multret = multiquery_retriver.invoke(query)

C:\Users\Ayon\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [64]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a helpful AI assistant.

        Answer only using the provided context.

        If the answer is not in the context, say:
        "I couldn't find that information in the provided documents."

        Context:
        {context}

        Question:
        {input}""",
        input_variables=['context', 'input']
)

In [65]:
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

parser = StrOutputParser()
model_qs = ChatGroq(
    model = 'llama-3.1-8b-instant'
)

chain = prompt | model_qs | parser

chain.invoke(
    {
        'context': multret,
        'input':query
    }
)

"To install Python, you can either use the Python install manager or the standalone version. I recommend going with the standalone version because it's going to be a little bit simpler, and you can install this on Mac, Windows, or Linux. \n\n1. Open up your browser and go to python.org.\n2. Go to the download section. \n3. For downloads, you can just find the most recent download here, so Python install manager, or you can install the standalone version.\n4. I recommend that you go with the standalone version because it's going to be a little bit simpler, and you can install this on Mac, Windows, or Linux.\n5. Press the download button for the standalone version.\n6. Once it's downloaded, double-click on it so that you can install Python and have that on your system.\n7. When prompted, check the checkbox that says add Python exe to path. It's going to make your life a lot easier later. \n8. Choose all of the default options, run through the installation, and you'll be good to go."